In [1]:
import random
import secrets

In [2]:
def generate_small_prime() -> list[int]:
    small_prime = []

    for number in range(2, 10000):
        i = 2
        is_prime = True
        while i ** 2 <= number:
            if number % i == 0:
                is_prime = False
                break
            i += 1

        if is_prime:
            small_prime.append(number)

    return small_prime


def check_small_number(number: int) -> bool:
    small_prime = generate_small_prime()

    for num in small_prime:
        if (number % num == 0):
            return False
    return True

In [3]:
def is_probable_prime(number: int) -> bool:
    if (number <= 1):
        return False
    elif (number <= 3):
        return True

    if (not check_small_number(number)):
        return False
    
    k = 0
    while (((number - 1) % (2**(k + 1))) == 0):
        k += 1
    
    m = (number - 1) // (2 ** k)

    for _ in range(64):
        is_prime = False
        a = random.randint(2, number - 2)
        b = pow(a, m, number)

        if (b == 1 or b == number - 1):
            continue
        
        for _ in range(k - 1):
            b = pow(b, 2, number)
            if (b == number - 1):
                is_prime = True
                break
        if (not is_prime):
            return False
    
    return True

In [4]:
def generate_big_number(bit: int) -> int:
    number = secrets.randbits(bit)
    return number


def generate_big_prime_number(bit: int) -> int:
    number = generate_big_number(bit)
    while (not is_probable_prime(number)):
        number = generate_big_number(bit)
    
    return number

In [5]:
def is_coprime(num1: int, num2: int) -> bool:
    while (num2 != 0):
        temp = num1
        num1 = num2
        num2 = temp % num2
    
    if (num1 == 1):
        return True
    
    return False

In [6]:
def find_e(t: int) -> int:
    # (e * d) % t = 1
    # e must be less than t and coprime with t
    
    e = 65537

    while (not is_coprime(t, e)):
        e = random.randint(65537, t)

    return e

In [7]:
def generate_keys(bit: int) -> tuple:
    p = generate_big_prime_number(bit)
    q = generate_big_prime_number(bit)

    n = p * q
    t = (p - 1) * (q - 1)

    e = find_e(t)
    # (e * d) % t = 1
    d = pow(e, -1, t)

    return (e, d, n)

In [ ]:
def string_to_block_int(message: str, n: int) -> list[int]:
    encoded_text = message.encode('utf-8')
    block_size = (n.bit_length() - 1) // 8
    blocks = []

    encoded_text += b'\x80'
    padding_len = block_size - (len(encoded_text) % block_size)

    if padding_len != block_size:
        encoded_text += b'\x00' * padding_len


    for i in range(0, len(encoded_text), block_size):
        block = encoded_text[i : i + block_size]
        blocks.append(int.from_bytes(block, 'big'))

    return blocks


In [ ]:
def encrypt_message(n: int, e: int, message: str) -> list[int]:
    blocks = string_to_block_int(message, n)
    ciphers = []

    for block in blocks:
        ciphers.append(pow(block, e, n))

    return ciphers

In [ ]:
def decrypt_message(n: int, d: int, ciphers: list[int]) -> str:
    messages_int = []

    # message = cipher ** d % n
    for cipher in ciphers:
        messages_int.append(pow(cipher, d, n))

    bytes_list = bytearray()
    block_size = (n.bit_length() - 1) // 8

    for i in range(len(messages_int)):
        block_bytes = messages_int[i].to_bytes(block_size, 'big')
        bytes_list.extend(block_bytes)

    padding_index = bytes_list.rindex(0x80)
    bytes_list = bytes_list[:padding_index]

    return bytes_list.decode('utf-8')

In [ ]:
def test(message: str, e: int, d: int, n: int) -> bool:
    ciphers = encrypt_message(n, e, message)
    original = decrypt_message(n, d, ciphers)
    
    if (message == original):
        return True
    return False

In [11]:
print(generate_keys(1000))

(65537, 8114575254496156038668544425773244977753516606921380488234365461597177353191218279559789797819155994896231850700393258855840053357549834985850007694849814994380649418633189225305342466194858065743265422105887408851116233686891521608256921558164640651373781336541794997332946860642163558037765343074576658547631365249685375351773821937003024671310331220342607990812034661119817145179755224048811151652786421742748293847585206546661554733609366151630192699044023743947273080868130444132746085064730205380226257710583937922744625186332688174327470618595006206569895653028802037356020921989039559998353793, 15801661519949920615249454643645852209390349661797846176122882461943075685517541892363985855880969438048264650107017471568420430155808751610983567205389164322885889791976328101584793617798615749654327320414605351771671508160386559252424123255309346437946323204663188790379121628462501057885040180926424098495208082958828693768021666702971499561355967848124125032661650547573033727100963230